In [ ]:
import streamlit as st

from typing import TypedDict, List, Dict, Annotated, Optional
import operator

from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

In [ ]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from langchain_google_genai import GoogleGenerativeAI
llm =  GoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:
config = {
    "system_prompt": "", 
    "human_prompt": "", 
    "output_mapping": "output",
    "schema": None, 
    "static_returns": {},
    "requirements": []
}

In [ ]:
class TaskStep(TypedDict):
    total_seq: int
    agent_role: str
    role_seq: int
    instruction: str
    output_summary: str

class BaseAgentState(TypedDict):
    initial_intent: str
    curr_seq: int
    task_seq: Dict[str, int]
    curr_agent: str
    task_history: Annotated[List[TaskStep], operator.add]
    latest_raw_output: str

In [ ]:
def write_history(state: AgentState, result: str, task_name: str) -> dict:
    # 1. 순번 계산 (파이썬이 직접 수행)
    new_total_seq = state.get("curr_seq", 0) + 1
    
    # 2. 에이전트별 카운트 업데이트
    new_task_seq = dict(state.get("task_seq", {}))
    new_task_seq[task_name] = new_task_seq.get(task_name, 0) + 1
    
    # 3. 장부에 기록할 정형 데이터 조립
    new_step = {
        "total_seq": new_total_seq,
        "agent_role": task_name,
        "role_seq": new_task_seq[task_name],
        "output_summary": result  # LLM이 뱉은 요약 텍스트
    }
    
    # 4. 업데이트할 변수들 리턴
    return {
        "curr_seq": new_total_seq,
        "task_seq": new_task_seq,
        "task_history": [new_step],
        "curr_agent": task_name,
        "latest_raw_output": result
    }

In [ ]:
# output_mapping: 결과물이 하나 뿐이라면, 결과물을 state 내에 등록할 때의 변수명
# schema: 결과물을 정형화된 형태로 할 경우, 기존에 정해둔 데이터 프레임
# static_returns: 

agent_registry = {
    "worker": {
        "system_prompt": "당신은 작업자입니다. 작업 주제, 작업 맥락 (주어진 경우 이전 작업물, 이전 작업물)에 대한 피드백을 보고 훌륭한 작업물을 생성하세요.",
        "human_prompt": "[작업 주제]: {topic}\n[작업 맥락]: {context}\n[이전 작업물]: {prev_draft}\n[피드백]: {human_feedback}",
        "output_mapping": "latest_raw_output", 
        "requirements": {"topic": str, "context":str, "prev_draft":str, "human_feedback":str}
    },
    "advisor": {
        "system_prompt": "당신은 검토자입니다. 작업 주제, 작업 맥락, 작업자의 작업물을 보고 비판적으로 검토하세요.",
        "human_prompt": "[작업 주제]: {topic}\n[작업 맥락]: {context}\n[작업물]: {draft}",
        "output_mapping": "latest_raw_output",
        "requirements": {"topic": str, "context":str, "draft":str}
    },
    "summary": {
        "system_prompt": "당신은 요약자입니다. 작업물을 보고 가능한 디테일 살리되 길이를 최소화한 요약문을 작성하세요.",
        "human_prompt": "[작업물]: {latest_raw_output}",
        "action": write_history,
        "requirements": {"latest_raw_output": str}
    },
}

In [ ]:
def initiate_state(agent_registry: dict):
    requirements = {}
    for registry in agent_registry.values():
        requirements.update(registry["requirements"])

    DynamicState = type("AgentState", (BaseAgentState,), requirements)

    return DynamicState

In [ ]:
def call_task(task_name: str, state: AgentState) -> dict:
    # 1. 설정 불러오기
    config = agent_registry[task_name]

    # 2. 프롬프트 준비
    prompt = ChatPromptTemplate.from_messages([
        ("system", config["system_prompt"]),
        ("human", config["human_prompt"])
    ])

    # 3. 파서 및 체인 연결
    if config["schema"]:
        parser = PydanticOutputParser(pydantic_object=config["schema"])
        prompt = prompt.partial(format=parser.get_format_instructions())
        chain = prompt | llm | parser
    else:
        chain = prompt | llm

    # 4. LLM 실행
    result = chain.invoke(**state)

    # 5. 리턴할 State 딕셔너리 조립
    # schema가 있으면 Pydantic 객체를 딕셔너리로 변환해서 넣음 (예: is_sufficient, parsed_intent 등)
    # schema가 없으면 output_mapping 이름으로 문자열을 넣음 (예: worker_output)
    updates = {}
    if config.get("schema"):
        updates.update(result.model_dump())
    elif config.get("output_mapping"):
        updates[config["output_mapping"]] = result

    if "action" in config:
        action_updates = config["action"](state)
        updates.update(action_updates)

    updates.update(config.get("static_returns", {}))

    return updates

In [1]:
temp_dict = {"hi":"bye"}
inputs = {**temp_dict}
inputs

{'hi': 'bye'}

In [2]:
print(temp_dict.get("bye"))

None
